In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip -q install torch torchvision numpy pandas scikit-learn tqdm Pillow
import os
from pathlib import Path
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss, f1_score
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)


In [4]:
DATA_ROOT = Path('/content/drive/MyDrive/OneDrive_2025-04-08/ImagingIQ-IIT-BHU-DataScientistScreener/data')
TRAIN_IMAGES_DIR = DATA_ROOT / 'images' / 'train'
VAL_IMAGES_DIR = DATA_ROOT / 'images' / 'val'
TEST_IMAGES_DIR = DATA_ROOT / 'images' / 'test_public'
LABELS_TRAIN_CSV = DATA_ROOT / 'labels_train.csv'
LABELS_VAL_CSV = DATA_ROOT / 'labels_val.csv'
METADATA_CSV = DATA_ROOT / 'metadata.csv'
SPLITS_CSV = DATA_ROOT / 'splits.csv'
for p in [TRAIN_IMAGES_DIR, VAL_IMAGES_DIR, TEST_IMAGES_DIR, LABELS_TRAIN_CSV, LABELS_VAL_CSV, METADATA_CSV, SPLITS_CSV]:
    print(p, '->', 'OK' if p.exists() else 'MISSING')


/content/drive/MyDrive/OneDrive_2025-04-08/ImagingIQ-IIT-BHU-DataScientistScreener/data/images/train -> OK
/content/drive/MyDrive/OneDrive_2025-04-08/ImagingIQ-IIT-BHU-DataScientistScreener/data/images/val -> OK
/content/drive/MyDrive/OneDrive_2025-04-08/ImagingIQ-IIT-BHU-DataScientistScreener/data/images/test_public -> OK
/content/drive/MyDrive/OneDrive_2025-04-08/ImagingIQ-IIT-BHU-DataScientistScreener/data/labels_train.csv -> OK
/content/drive/MyDrive/OneDrive_2025-04-08/ImagingIQ-IIT-BHU-DataScientistScreener/data/labels_val.csv -> OK
/content/drive/MyDrive/OneDrive_2025-04-08/ImagingIQ-IIT-BHU-DataScientistScreener/data/metadata.csv -> OK
/content/drive/MyDrive/OneDrive_2025-04-08/ImagingIQ-IIT-BHU-DataScientistScreener/data/splits.csv -> OK


In [5]:
labels_train = pd.read_csv(LABELS_TRAIN_CSV)
labels_val = pd.read_csv(LABELS_VAL_CSV)
metadata = pd.read_csv(METADATA_CSV)
print('labels_train:', labels_train.shape)
print('labels_val:', labels_val.shape)
print('metadata:', metadata.shape
assert {'image_id','class'}.issubset(labels_train.columns)
assert {'image_id','class'}.issubset(labels_val.columns)
triage_map = {
    0: 'Routine',
    1: 'Non-urgent abnormal',
    2: 'Urgent',
    3: 'Critical'
}
labels_train['triage'] = labels_train['class'].map(triage_map)
labels_val['triage'] = labels_val['class'].map(triage_map)
labels_train.head()

labels_train: (8000, 2)
labels_val: (1000, 2)
metadata: (10000, 9)


,image_id,class,triage
0,00000032_000.png,1,Non-urgent abnormal
1,00000032_001.png,2,Urgent
2,00000032_002.png,0,Routine
3,00000032_003.png,0,Routine
4,00000032_004.png,2,Urgent


In [6]:
IMG_SIZE = 224
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [7]:
class CXRDataset(Dataset):
    def __init__(self, images_dir, labels_df, transform=None):
        self.images_dir = Path(images_dir)
        self.df = labels_df[['image_id','class']].copy()
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['image_id']
        label = int(row['class'])
        img_path = self.images_dir / image_id
        if not img_path.exists():
            raise FileNotFoundError(f"Missing image: {img_path}")
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, image_id
BATCH_SIZE = 32
train_ds = CXRDataset(TRAIN_IMAGES_DIR, labels_train, transform=train_tf)
val_ds = CXRDataset(VAL_IMAGES_DIR, labels_val, transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))

Train batches: 250 Val batches: 32


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 4)
model = model.to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=3e-4)


Device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 41.1MB/s]


In [9]:
def compute_metrics(y_true, y_proba):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    y_bin = np.isin(y_true, [2,3]).astype(int)
    pos_score = y_proba[:,2] + y_proba[:,3]
    metrics = {}
    metrics['auprc'] = average_precision_score(y_bin, pos_score)
    metrics['auroc'] = roc_auc_score(y_bin, pos_score)
    metrics['log_loss'] = log_loss(y_true, y_proba, labels=[0,1,2,3])
    y_pred = np.argmax(y_proba, axis=1)
    metrics['macro_f1'] = f1_score(y_true, y_pred, average='macro')
    y_true_onehot = np.zeros_like(y_proba)
    y_true_onehot[np.arange(len(y_true)), y_true] = 1.0
    metrics['brier'] = float(np.mean(np.sum((y_proba - y_true_onehot)**2, axis=1)))
    return metrics
EPOCHS = 3
best_auprc = -1
os.makedirs('models', exist_ok=True)
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0.0
    for imgs, labels, _ in tqdm(train_loader, desc=f"train {epoch}/{EPOCHS}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_loader.dataset)
    model.eval()
    val_loss = 0.0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels, _ in tqdm(val_loader, desc=f"val {epoch}/{EPOCHS}"):
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            val_loss += loss.item() * imgs.size(0)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.cpu().numpy())
    val_loss /= len(val_loader.dataset)
    y_proba = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels)
    m = compute_metrics(y_true, y_proba)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
          f"AUPRC(2|3)={m['auprc']:.4f} AUROC(2|3)={m['auroc']:.4f} "
          f"log_loss={m['log_loss']:.4f} macro_f1={m['macro_f1']:.4f} brier={m['brier']:.4f}")
    if m['auprc'] > best_auprc:
        best_auprc = m['auprc']
        torch.save({'state_dict': model.state_dict(), 'img_size': IMG_SIZE}, 'models/best.pt')
        print('Saved models/best.pt')

val 1/3: 100%|██████████| 32/32 [03:23<00:00,  6.36s/it]


Epoch 1: train_loss=1.0325 val_loss=1.1443 AUPRC(2|3)=0.5345 AUROC(2|3)=0.8516 log_loss=1.0753 macro_f1=0.3077 brier=0.5913
Saved models/best.pt


val 2/3: 100%|██████████| 32/32 [01:39<00:00,  3.10s/it]


Epoch 2: train_loss=0.9366 val_loss=1.0260 AUPRC(2|3)=0.5573 AUROC(2|3)=0.8620 log_loss=0.9441 macro_f1=0.3895 brier=0.5344
Saved models/best.pt


val 3/3: 100%|██████████| 32/32 [01:48<00:00,  3.39s/it]

Epoch 3: train_loss=0.8990 val_loss=1.0345 AUPRC(2|3)=0.5535 AUROC(2|3)=0.8707 log_loss=0.9539 macro_f1=0.3599 brier=0.5441


In [10]:
print('best vvalidation AUPRC(urgent|critical) :', best_auprc)

best vvalidation AUPRC(urgent|critical) : 0.5573349314173581


In [11]:
ckpt = torch.load('models/best.pt', map_location='cpu')
model.load_state_dict(ckpt['state_dict'])
model.eval()
meta = pd.read_csv(METADATA_CSV)
image_ids = meta['image_id'].tolist()
rows = []
for image_id in tqdm(image_ids):
    img_path = TEST_IMAGES_DIR / image_id
    if not img_path.exists():
        continue
    img = Image.open(img_path).convert('RGB')
    inp = val_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inp)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    probs = np.clip(probs, 0.0, 1.0)
    s = probs.sum()
    probs = probs / s if s > 0 else np.array([0.25, 0.25, 0.25, 0.25])
    rows.append({'image_id': image_id, 'p0': float(probs[0]), 'p1': float(probs[1]), 'p2': float(probs[2]), 'p3': float(probs[3])})

pred_df = pd.DataFrame(rows)
print(pred_df.head())

100%|██████████| 10000/10000 [08:19<00:00, 20.01it/s]

           image_id        p0        p1        p2        p3
0  00000177_000.png  0.264658  0.548143  0.174345  0.012854
1  00000177_001.png  0.317073  0.537637  0.135475  0.009815
2  00000177_002.png  0.405948  0.383095  0.145932  0.065026
3  00000177_003.png  0.400487  0.578969  0.008318  0.012226
4  00000177_004.png  0.727062  0.259564  0.006567  0.006808


In [12]:
os.makedirs('outputs', exist_ok=True)
out_csv = Path('outputs/test_public_predictions.csv')
pred_df[['image_id','p0','p1','p2','p3']].to_csv(out_csv, index=False)
print('Saved:', out_csv)
os.makedirs('submission/models', exist_ok=True)
os.makedirs('submission/outputs', exist_ok=True)
!cp models/best.pt submission/models/
!cp outputs/test_public_predictions.csv submission/outputs/
!zip -r submission.zip submission
print('Created submission.zip')

Saved: outputs/test_public_predictions.csv
  adding: submission/ (stored 0%)
  adding: submission/models/ (stored 0%)
  adding: submission/models/best.pt (deflated 7%)
  adding: submission/outputs/ (stored 0%)
  adding: submission/outputs/test_public_predictions.csv (deflated 57%)
Created submission.zip
